# E1 Exercises: Data Cleaning

> Northbridge Community Clinic appointment export · ~90 minutes · needs `notebooks/C1_Data_Cleaning.ipynb`

## Objective → task map

Every objective the chapter states is assessed by at least one task below. Where the
chapter's own embedded exercises already cover one, that is said here rather than
duplicated.

| # | Objective | Assessed by |
|---|---|---|
| 1 | Load and inspect datasets | Task 1 |
| 2 | Identify and handle missing values | Tasks 3, 4 |
| 3 | Detect and treat outliers | Tasks 6, 7 |
| 4 | Clean and format data properly | Tasks 2, 3, 7 |
| 5 | Convert data types appropriately | Tasks 4, 5 |
| 6 | Check for duplicate rows (C1 checklist) | Task 2 |

## Before you start

**Everything here is solvable with C1 alone.** `read_csv`, `.info()`, `.describe()`, `.isnull()`, `.value_counts()`, `.duplicated()`, `.drop_duplicates()`, `.fillna()`, `.dropna()`, `pd.to_numeric`, `pd.to_datetime`, `.astype()`, `.str` accessors, `.apply()`, the IQR rule and `scipy.stats.zscore`. Two of these (`.duplicated()` and `pd.to_datetime`) appear on C1's checklist and conversion table without a worked example; the syntax for both is in the briefs below.

Nothing below needs a method the chapter did not cover. Where a task is hard, it is
hard because of a decision you have to make and justify, not because of syntax.

**Marks.** 100 in total. Task 7 carries 40 and is the one that separates a student who followed the C1 recipe from one who understood why each step is there.

**Dataset path used in this notebook:** `../data/clinic_visits_raw.csv`

---

## Task 1: First look

**Warm-up · 5 marks · Load and inspect datasets**

Load `clinic_visits_raw.csv` and inspect it. Report the shape, the dtypes, and the
missing-value count per column.

Then answer two questions in a comment or a markdown cell:

**(a)** Two columns hold numbers but were loaded as `object` (text). Which two, and
what does that tell you before you have looked at a single value?

**(b)** `patient_age` and `wait_minutes` both have missing values, but
`.isnull().sum()` reports missing values for only one of them. Which one is lying to
you, and what is the clue that gives it away?

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../../00-data/clinic_visits_raw.csv')

# Your code here

**Your answer to Task 1:**

*(write here)*

---

## Task 2: Duplicate rows

**Warm-up · 5 marks · Clean and format data properly / C1 checklist item**

Part of this export was submitted twice.

1. How many rows are exact duplicates of an earlier row?
2. Does `visit_id` agree with that count?
3. Remove them, then report the mean `billed_amount` before and after.
4. Why must this step come before you compute any statistic at all?

In [ ]:
# Your code here

**Your answer to Task 2:**

*(write here)*

---

## Task 3: The department column, and the order of operations

**Warm-up · 10 marks · Identify and handle missing values / clean and format**

`department` has missing values. C1 taught you to fill a categorical column with its
mode. Do exactly that, but in two different orders, and compare.

1. How many distinct values does pandas see in `department`? How many departments do
   you think the clinic actually has?
2. **Order A:** take the mode of the column as loaded. Write down which department it is.
3. **Order B:** normalise whitespace and capitalisation first, *then* take the mode.
   Write down which department it is now.
4. Are they the same department? If not, how many patients would Order A have
   assigned to the wrong one? Fix `df['department']` using the correct order.

In [ ]:
# Your code here

**Your answer to Task 3:**

*(write here)*

---

## Task 4: The missing values that are not missing

**Core · 10 marks · Convert data types / handle missing values**

In Task 1 you found that `.isnull()` reports zero missing `wait_minutes`.

1. Convert the column with `pd.to_numeric(..., errors='coerce')`. How many values
   are actually missing?
2. Before you overwrite anything, find out what those non-numeric values *were*.
   Print them with their counts. (Why does it matter what they say?)
3. Impute the missing wait times. Choose mean or median, and justify the choice in
   one sentence using a number you have computed, not a rule you have memorised.

In [ ]:
# Your code here

**Your answer to Task 4:**

*(write here)*

---

## Task 5: Three receptionists, three date formats

**Core · 15 marks · Convert data types appropriately**

`visit_date` was typed by three different people.

1. Run `pd.to_datetime(df['visit_date'], errors='coerce')` and count how many rows
   became `NaT`. Before you look: how many do you *expect* to lose?
2. Look at which rows survived and which did not. Explain why that particular group
   survived. (This is the part that catches people.)
3. Recover **all** rows. State how many are unparseable when handled properly.
4. What is the general lesson about `errors='coerce'` on a date column?

In [ ]:
# Your code here

**Your answer to Task 5:**

*(write here)*

---

## Task 6: A sentinel hiding inside an outlier check

**Core · 15 marks · Detect and treat outliers**

The billing system writes `9999` into `billed_amount` when a claim is rejected.
Nobody told the analytics team.

1. Run `.describe()` on `billed_amount`. What in that output should have caught this?
2. Count the sentinel rows.
3. Run the IQR outlier check **with** the sentinel present. Report the upper bound
   and the number of rows flagged.
4. Replace the sentinel with `NaN` and run the same check again. Report both numbers.
5. Two separate things went wrong in step 3. Name both. One is obvious; the other is
   the reason this task is worth 15 marks.

In [ ]:
# Your code here

**Your answer to Task 6:**

*(write here)*

---

## Task 7: The colleague's clean weight column

**Judgement · 40 marks · Detect and treat outliers / clean and format data properly**

A colleague cleaned `patient_weight` for you. They copied C1's `clean_length`
function, swapped `miles` for the weight units, and got this:

```python
def clean_weight(val):
    if pd.isna(val):
        return np.nan
    text = str(val).lower()
    for unit in ('kgs', 'kg', 'lbs', 'lb'):
        text = text.replace(unit, '')
    try:
        return float(text.strip())
    except ValueError:
        return np.nan
```

Their notebook runs without a single error. Their mean patient weight is 79.8 kg,
which is a perfectly ordinary figure. Their IQR check flags 50 outliers, so they
capped them at the upper bound, exactly as C1 demonstrated, and moved on.

**Their column is wrong. Every number they produced is plausible and none of it is right.**

1. Reproduce their result: the cleaned column, the mean, the IQR bounds, the 50
   flagged rows.
2. Print the **raw strings** behind those 50 flagged rows. What do they have in
   common that the cleaned numbers no longer show?
3. Produce a correct `patient_weight`, and report the corrected mean and the number
   of genuine outliers.
4. **Written answer, 100 to 150 words.** State what capping would have done to those
   50 patients specifically, why no error message could ever have warned your
   colleague, and give a general rule for when IQR capping is safe to apply. Marks
   are for the reasoning, not for the code.

*No new methods are needed. Everything here is in C1.*

In [ ]:
# Your code here

**Your answer to Task 7:**

*(write here)*

---

## Before you submit

Work through C1's own summary checklist against your cleaned `df`:

- [ ] Loaded and inspected the data structure
- [ ] Identified and documented missing values (including the ones `.isnull()` missed)
- [ ] Applied an appropriate strategy for each column with missing data
- [ ] Detected outliers using visual and statistical methods
- [ ] Handled outliers appropriately, and can say *why* for each decision
- [ ] Cleaned messy text and mixed-format columns
- [ ] Converted every column to an appropriate data type
- [ ] Checked for and removed duplicate rows

Then run `df.info()` one last time. Every column should have the dtype its meaning
implies, and you should be able to explain every remaining missing value.